# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane: Growth / Recovery / Momentum Prediction** (confirmed — same month `2026-03`, same Mar 1–15 / Mar 16–31 feature/label split, same five honest features, and the same `is_declining_next_half` target as `w03_data_contract.ipynb` and `w04_baseline_score.ipynb`).

> Skills loaded: `training-honest-models/SKILL.md`, `flyrank/flyrank-data/SKILL.md` (per `skills/README.md`).


## 1. Method choice and why

My target, `is_declining_next_half`, is a **yes/no label observed after the fact** (impressions dropped >20% in the second half of March vs. the first half). Per `training-honest-models/SKILL.md`'s method table, that question shape points to **Logistic Regression first, then Random Forest** — readable, then stronger.

- **Logistic Regression** — a coefficient-per-feature model I can read and explain in one sentence per feature. It sets the honest floor: can a straight line through my 5 features beat the rule?
- **Random Forest** — adds non-linear splits and feature interactions (e.g. "low CTR AND few active days" might matter together, not separately) without needing me to hand-engineer interaction terms.
- **Not Gradient Boosting** — with only 5 features and ~tens of thousands of rows, a boosted model would mostly re-discover what the Random Forest already finds, at higher overfitting risk and less readability. Simplicity is a feature, not a shortcut (per the skill): I only reach for more complexity if it earns its keep in the comparison table below.

Both models are trained on the **same 5 honest features** as my Week-4 baseline (`impressions_first_half`, `clicks_first_half`, `ctr_first_half`, `avg_position_first_half`, `active_days_first_half`) — no new columns, so any improvement is genuinely "a model beats a rule," not "a model saw more data."


In [1]:
# ---- Setup: same DuckDB + HF pattern as w03/w04, columns already verified there ----
%pip -q install duckdb scikit-learn
import duckdb
import pandas as pd
import numpy as np
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN (plain Read token): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # same mid-panel month as w03/w04 -- never the _sample (sealed final) month
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Confirmed real column names (verified against DESCRIBE output in w03_data_contract.ipynb)
COLS = {
    "impressions": "gsc_impressions",
    "clicks": "gsc_clicks",
    "position": "gsc_avg_position",
    "ga4_flag": "ga4_data_available",
}
print("Using verified columns:", COLS)


Using verified columns: {'impressions': 'gsc_impressions', 'clicks': 'gsc_clicks', 'position': 'gsc_avg_position', 'ga4_flag': 'ga4_data_available'}


## 2. Split design

**Grouped by client, not random rows.** I split on `client_hash_id` (70% of clients → train, 30% → test, `test_size=0.3`, `random_state=42` — the same split recipe I already validated in `w03_data_contract.ipynb`'s leakage-trap check). A random row split would let two content items from the *same* client sit in train and test at once, leaking that client's template/strategy quirks across the split — the grouped split closes that.

**Time is already handled by the feature engineering, not the split.** The label (`is_declining_next_half`) is built from Mar 16–31, strictly *after* the Mar 1–15 feature window — every row is already a genuine past→future prediction before any train/test split happens. The client split only needs to prevent *client* leakage on top of that.

**Fair comparison with the Week-4 baseline:** `w04_baseline_score.ipynb` scored the *whole* month's data at once — no held-out test set. That's not a fair fight against a model evaluated on unseen clients. So here I rebuild the baseline rule using thresholds fit **only on the train clients** (the same weighted-CTR-per-position-tier logic as Week 4), then score **only the test clients** with it — baseline and models now see the exact same unseen split.


In [ ]:
# ---- Rebuild the same first-half feature frame as w03/w04 (no future-window inputs) ----
feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_first_half,
            SUM({COLS['clicks']})      AS clicks_first_half,
            AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
            COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        f.impressions_first_half,
        f.clicks_first_half,
        ROUND(100.0 * f.clicks_first_half / NULLIF(f.impressions_first_half, 0), 2) AS ctr_first_half,
        f.avg_position_first_half,
        f.active_days_first_half,
        COALESCE(s.impressions_second_half, 0) AS impressions_second_half,
        CASE
            WHEN f.impressions_first_half > 0
                 AND (COALESCE(s.impressions_second_half, 0) - f.impressions_first_half)
                     / f.impressions_first_half < -0.20
            THEN 1 ELSE 0
        END AS is_declining_next_half
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    WHERE f.impressions_first_half > 0
""").df()

HONEST_FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]
feature_frame = feature_frame.dropna(subset=HONEST_FEATURES).reset_index(drop=True)

# ---- FIX: lock in a fixed row order. DuckDB does not guarantee row order, so without
# this, which rows land in a tied "top 10" could silently change between runs. ----
feature_frame = feature_frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

print(f"Rows: {len(feature_frame):,} (client, content) pairs")
print(f"Base rate of is_declining_next_half: {feature_frame['is_declining_next_half'].mean():.1%}")

# ---- Grouped client split (same recipe as w03's quick LR check) ----
from sklearn.model_selection import train_test_split

train_clients, test_clients = train_test_split(
    feature_frame["client_hash_id"].unique(), test_size=0.3, random_state=42
)
train_df = feature_frame[feature_frame["client_hash_id"].isin(train_clients)].reset_index(drop=True)
test_df  = feature_frame[feature_frame["client_hash_id"].isin(test_clients)].reset_index(drop=True)

print(f"Train: {len(train_df):,} rows, {train_df['client_hash_id'].nunique()} clients "
      f"(base rate {train_df['is_declining_next_half'].mean():.1%})")
print(f"Test:  {len(test_df):,} rows, {test_df['client_hash_id'].nunique()} clients "
      f"(base rate {test_df['is_declining_next_half'].mean():.1%})")


## 3. Train + compare vs my baseline

Same data (this feature frame), same test clients, same primary metric (**precision@10**, matching Week 4) — plus **precision@50** and **ROC-AUC** for a fuller picture, and the test-set **base rate** as the floor any method must clear. The baseline rule is refit on train-only thresholds and scored on the test clients only, so all three rows below are judged on identical unseen data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

MIN_IMPRESSIONS = 50  # same floor as w04

# ---- FIX: kind="stable" means tied scores always break the same way, run to run. ----
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = np.asarray(y_true)[order][:k]
    return top_k.mean()

# ---- Baseline rule, refit on TRAIN only, scored on TEST only (fair vs. w04's whole-month fit) ----
def position_tier(p):
    if pd.isna(p) or p <= 0: return "no_data"
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

train_df = train_df.copy(); test_df = test_df.copy()
train_df["position_tier"] = train_df["avg_position_first_half"].apply(position_tier)
test_df["position_tier"]  = test_df["avg_position_first_half"].apply(position_tier)

train_tier_ctr = (
    train_df.groupby("position_tier")
    .apply(lambda g: 100 * g["clicks_first_half"].sum() / g["impressions_first_half"].sum())
)
test_df["expected_ctr_for_tier"] = test_df["position_tier"].map(train_tier_ctr)

test_df["has_volume"] = (test_df["impressions_first_half"] >= MIN_IMPRESSIONS).astype(int)
test_df["low_ctr"] = (test_df["ctr_first_half"] < test_df["expected_ctr_for_tier"]).astype(int)
test_df["baseline_score"] = (
    test_df["has_volume"] * test_df["low_ctr"] * test_df["impressions_first_half"]
)

y_test = test_df["is_declining_next_half"].values
baseline_p10 = precision_at_k(y_test, test_df["baseline_score"].values, 10)
baseline_p50 = precision_at_k(y_test, test_df["baseline_score"].values, 50)
baseline_auc = roc_auc_score(y_test, test_df["baseline_score"].values)

# ---- Logistic Regression -- FIX: scale features first. Unscaled, impressions_first_half
# (scale: thousands) drowns out ctr_first_half (scale: 0-100), leaving LR coefficients
# near zero and predicted scores nearly tied -- which is what was making precision@10
# unstable between runs. ----
X_train, X_test = train_df[HONEST_FEATURES], test_df[HONEST_FEATURES]
y_train = train_df["is_declining_next_half"].values

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
lr_scores = lr.predict_proba(X_test_scaled)[:, 1]
lr_p10 = precision_at_k(y_test, lr_scores, 10)
lr_p50 = precision_at_k(y_test, lr_scores, 50)
lr_auc = roc_auc_score(y_test, lr_scores)

# ---- Random Forest (tree-based, doesn't need scaling) ----
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1
).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p10 = precision_at_k(y_test, rf_scores, 10)
rf_p50 = precision_at_k(y_test, rf_scores, 50)
rf_auc = roc_auc_score(y_test, rf_scores)

# ---- The comparison table (non-negotiable, per the skill) ----
comparison = pd.DataFrame([
    {"method": "base_rate (floor)",        "precision_at_10": y_test.mean(), "precision_at_50": y_test.mean(), "roc_auc": 0.500},
    {"method": "Week-4 rule baseline",      "precision_at_10": baseline_p10, "precision_at_50": baseline_p50, "roc_auc": baseline_auc},
    {"method": "Logistic Regression",       "precision_at_10": lr_p10,       "precision_at_50": lr_p50,       "roc_auc": lr_auc},
    {"method": "Random Forest",             "precision_at_10": rf_p10,       "precision_at_50": rf_p50,       "roc_auc": rf_auc},
]).round(3)
print(f"Test set: {len(test_df):,} rows, {test_df['client_hash_id'].nunique()} held-out clients\n")
comparison


## 4. Errors and interpretation

*Filled after you run cell above and see real numbers — the code below reads whichever of Logistic Regression / Random Forest scored higher on precision@10 as "the model", and reports: which rows it gets wrong, what it leans on (coefficients + permutation importance, sanity-checked against the rule's own logic), and 3 concrete wrong cases with the reason they're hard. If Random Forest doesn't clearly beat Logistic Regression at precision@10, that itself is the finding — the extra complexity didn't earn its keep, and Logistic Regression is the one worth shipping.*


In [ ]:
from sklearn.inspection import permutation_importance

best_name, best_model, best_scores = max(
    [("Logistic Regression", lr, lr_scores), ("Random Forest", rf, rf_scores)],
    key=lambda t: precision_at_k(y_test, t[2], 10)
)
print(f"Reading errors for: {best_name} (higher precision@10 on the held-out test clients)\n")

# ---- What it leans on ----
print("Logistic Regression coefficients (direction + rough weight):")
for feat, coef in sorted(zip(HONEST_FEATURES, lr.coef_[0]), key=lambda t: -abs(t[1])):
    print(f"  {feat:28s} {coef:+.3f}")

print("\nRandom Forest permutation importance (drop in ROC-AUC when a feature is shuffled):")
perm = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42)
for feat, imp in sorted(zip(HONEST_FEATURES, perm.importances_mean), key=lambda t: -t[1]):
    print(f"  {feat:28s} {imp:+.4f}")

# ---- Where is it most wrong? by position tier and volume tier ----
test_df["pred_score"] = best_scores
test_df["predicted_top10"] = test_df["pred_score"].rank(ascending=False) <= 10

def volume_tier(imp):
    if imp < 10: return "low (<10)"
    if imp < 100: return "moderate (10-99)"
    if imp < 1000: return "good (100-999)"
    return "high (1000+)"
test_df["volume_tier"] = test_df["impressions_first_half"].apply(volume_tier)

error_by_tier = (
    test_df.assign(pred_label=(test_df["pred_score"] >= test_df["pred_score"].quantile(0.9)).astype(int))
    .assign(wrong=lambda d: d["pred_label"] != d["is_declining_next_half"])
    .groupby(["position_tier", "volume_tier"])
    .agg(n=("wrong", "size"), error_rate=("wrong", "mean"))
    .query("n >= 20")
    .sort_values("error_rate", ascending=False)
)
print("\nWhere the model is most wrong (top-decile prediction vs actual, by tier, n>=20):")
print(error_by_tier.head(10))

# ---- 3 concrete wrong cases ----
top10_pred = test_df.sort_values("pred_score", ascending=False).head(10)
wrong_top10 = top10_pred[top10_pred["is_declining_next_half"] == 0]
print(f"\n3 concrete wrong cases (flagged in the predicted top 10, did NOT actually decline):")
for i, row in wrong_top10.head(3).iterrows():
    print(f"- content {row.content_hash_id[:16]}...: score={row.pred_score:.3f}, "
          f"impressions_first_half={row.impressions_first_half:.0f}, "
          f"ctr_first_half={row.ctr_first_half:.2f}%, position={row.avg_position_first_half:.1f}, "
          f"active_days={row.active_days_first_half:.0f} "
          f"-- hard because: {'few active days, noisy CTR estimate' if row.active_days_first_half < 5 else 'looked at-risk on paper but held steady -- likely a case the 5 features genuinely cannot see (e.g. a mid-March content refresh not captured here)'}")
